# 03 - Data Cleaning

## Objective
Clean all five datasets and save the cleaned versions to `data/processed/`.

## Cleaning steps:
1. Missing values — identify, decide, and handle
2. Duplicates — identify and remove when justified
3. Data types — convert incorrect columns
4. Categories — normalize inconsistencies
5. Skill names — normalize naming variations
6. Outliers — identify and document decisions

## IMPORTANT
**NEVER modify files inside `data/raw/`.** Raw data must remain untouched.
---

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = "../data/raw"
OUT_PATH = "../data/processed"
os.makedirs(OUT_PATH, exist_ok=True)

# Load all datasets
df_attrition = pd.read_csv(f"{DATA_PATH}/employee_attrition.csv")
df_engagement = pd.read_csv(f"{DATA_PATH}/hr_performance_engagement.csv")
df_occupation = pd.read_csv(f"{DATA_PATH}/occupation_data.csv")
df_essential = pd.read_csv(f"{DATA_PATH}/essential_skills.csv")
df_software = pd.read_csv(f"{DATA_PATH}/software_skills.csv")

print("All 5 datasets loaded for cleaning.")

All 5 datasets loaded for cleaning.


---
## 1. Cleaning employee_attrition.csv
---

In [2]:
print("=== BEFORE CLEANING ===")
print(f"Shape: {df_attrition.shape}")
print(f"Missing values: {df_attrition.isnull().sum().sum()}")
print(f"Duplicates: {df_attrition.duplicated().sum()}")
print()

# Drop constant columns (not useful for analysis or modeling)
constant_cols = []
for col in df_attrition.columns:
    if df_attrition[col].nunique() <= 1:
        constant_cols.append(col)
        
print(f"Constant columns (single unique value): {constant_cols}")
for col in constant_cols:
    print(f"  {col}: value = {df_attrition[col].iloc[0]}")
    
print()
print("Dropping constant columns: EmployeeCount, Over18, StandardHours")
df_attrition_clean = df_attrition.drop(columns=["EmployeeCount", "Over18", "StandardHours"])

=== BEFORE CLEANING ===
Shape: (1470, 35)
Missing values: 0
Duplicates: 0

Constant columns (single unique value): ['EmployeeCount', 'Over18', 'StandardHours']
  EmployeeCount: value = 1
  Over18: value = Y
  StandardHours: value = 80

Dropping constant columns: EmployeeCount, Over18, StandardHours


In [3]:
# Check and handle missing values
print("=== MISSING VALUES ===")
missing = df_attrition_clean.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing if len(missing) > 0 else "No missing values found.")
print()

# If no missing values, note it
if df_attrition_clean.isnull().sum().sum() == 0:
    print("No imputation needed - dataset is complete.")

=== MISSING VALUES ===
No missing values found.

No imputation needed - dataset is complete.


In [4]:
# Check for duplicates
dup_count = df_attrition_clean.duplicated().sum()
print(f"Duplicate rows: {dup_count}")
if dup_count > 0:
    print(f"Removing {dup_count} duplicate rows.")
    df_attrition_clean = df_attrition_clean.drop_duplicates()
    print(f"Shape after dedup: {df_attrition_clean.shape}")
else:
    print("No duplicates to remove.")

Duplicate rows: 0
No duplicates to remove.


In [5]:
# Normalize categorical values - ensure consistent casing
print("=== CATEGORY NORMALIZATION ===")
print()

# Attrition: strip whitespace, ensure Yes/No
df_attrition_clean["Attrition"] = df_attrition_clean["Attrition"].str.strip()
print(f"Attrition values: {sorted(df_attrition_clean['Attrition'].unique())}")

# Department: strip whitespace
df_attrition_clean["Department"] = df_attrition_clean["Department"].str.strip()
print(f"Department values: {sorted(df_attrition_clean['Department'].unique())}")

# Gender: strip whitespace
df_attrition_clean["Gender"] = df_attrition_clean["Gender"].str.strip()
print(f"Gender values: {sorted(df_attrition_clean['Gender'].unique())}")

# MaritalStatus: strip whitespace
df_attrition_clean["MaritalStatus"] = df_attrition_clean["MaritalStatus"].str.strip()
print(f"MaritalStatus values: {sorted(df_attrition_clean['MaritalStatus'].unique())}")

# EducationField: strip whitespace
df_attrition_clean["EducationField"] = df_attrition_clean["EducationField"].str.strip()
print(f"EducationField values: {sorted(df_attrition_clean['EducationField'].unique())}")

# JobRole: strip whitespace
df_attrition_clean["JobRole"] = df_attrition_clean["JobRole"].str.strip()
print(f"JobRole values: {sorted(df_attrition_clean['JobRole'].unique())}")

=== CATEGORY NORMALIZATION ===

Attrition values: ['No', 'Yes']
Department values: ['Human Resources', 'Research & Development', 'Sales']
Gender values: ['Female', 'Male']
MaritalStatus values: ['Divorced', 'Married', 'Single']
EducationField values: ['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']
JobRole values: ['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative']


In [6]:
print("=== AFTER CLEANING ===")
print(f"Shape: {df_attrition_clean.shape}")
print(f"Missing values: {df_attrition_clean.isnull().sum().sum()}")
print(f"Duplicates: {df_attrition_clean.duplicated().sum()}")

# Save
df_attrition_clean.to_csv(f"{OUT_PATH}/employee_attrition_processed.csv", index=False)
print(f"Saved to: {OUT_PATH}/employee_attrition_processed.csv")

=== AFTER CLEANING ===
Shape: (1470, 32)
Missing values: 0
Duplicates: 0
Saved to: ../data/processed/employee_attrition_processed.csv


---
## 2. Cleaning hr_performance_engagement.csv
---

In [7]:
print("=== BEFORE CLEANING ===")
print(f"Shape: {df_engagement.shape}")
print(f"Missing values: {df_engagement.isnull().sum().sum()}")
print(f"Duplicates: {df_engagement.duplicated().sum()}")
print()

# Show missing values
missing = df_engagement.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Missing values by column:")
    print(missing)
else:
    print("No missing values.")

=== BEFORE CLEANING ===
Shape: (2845, 28)
Missing values: 0
Duplicates: 0

No missing values.


In [8]:
# Handle missing values - document decisions
df_engagement_clean = df_engagement.copy()

# For each column with missing values, decide strategy
for col in df_engagement_clean.columns:
    if df_engagement_clean[col].isnull().sum() > 0:
        null_count = df_engagement_clean[col].isnull().sum()
        null_pct = null_count / len(df_engagement_clean) * 100
        print(f"{col}: {null_count} missing ({null_pct:.1f}%)")
        
        if null_pct > 50:
            print(f"  -> Dropping column (>50% missing)")
            df_engagement_clean = df_engagement_clean.drop(columns=[col])
        elif pd.api.types.is_numeric_dtype(df_engagement_clean[col]):
            median_val = df_engagement_clean[col].median()
            print(f"  -> Filling with median ({median_val})")
            df_engagement_clean[col] = df_engagement_clean[col].fillna(median_val)
        else:
            mode_val = df_engagement_clean[col].mode()
            if len(mode_val) > 0:
                print(f"  -> Filling with mode ({mode_val.iloc[0]})")
                df_engagement_clean[col] = df_engagement_clean[col].fillna(mode_val.iloc[0])

In [9]:
# Remove duplicates
dup_count = df_engagement_clean.duplicated().sum()
print(f"Duplicate rows before: {dup_count}")
if dup_count > 0:
    df_engagement_clean = df_engagement_clean.drop_duplicates()
    print(f"Removed {dup_count} duplicates. Shape: {df_engagement_clean.shape}")
else:
    print("No duplicates to remove.")

# Normalize string columns
for col in df_engagement_clean.select_dtypes(include=["object", "str"]).columns:
    df_engagement_clean[col] = df_engagement_clean[col].str.strip()

print()
print("=== AFTER CLEANING ===")
print(f"Shape: {df_engagement_clean.shape}")
print(f"Missing values: {df_engagement_clean.isnull().sum().sum()}")

df_engagement_clean.to_csv(f"{OUT_PATH}/engagement_processed.csv", index=False)
print(f"Saved to: {OUT_PATH}/engagement_processed.csv")

Duplicate rows before: 0
No duplicates to remove.

=== AFTER CLEANING ===
Shape: (2845, 28)
Missing values: 0
Saved to: ../data/processed/engagement_processed.csv


---
## 3. Cleaning occupation_data.csv
---

In [10]:
print("=== BEFORE CLEANING ===")
print(f"Shape: {df_occupation.shape}")
print(f"Missing values: {df_occupation.isnull().sum().sum()}")
print(f"Duplicates: {df_occupation.duplicated().sum()}")

df_occupation_clean = df_occupation.copy()

# Drop exact duplicates
dup_count = df_occupation_clean.duplicated().sum()
if dup_count > 0:
    df_occupation_clean = df_occupation_clean.drop_duplicates()
    print(f"Removed {dup_count} duplicates.")

# Clean string columns
for col in df_occupation_clean.select_dtypes(include=["object", "str"]).columns:
    df_occupation_clean[col] = df_occupation_clean[col].str.strip()

# Standardize the O*NET-SOC Code format
df_occupation_clean["O*NET-SOC Code"] = df_occupation_clean["O*NET-SOC Code"].str.strip()

# Title: strip whitespace
df_occupation_clean["Title"] = df_occupation_clean["Title"].str.strip()

print()
print("=== AFTER CLEANING ===")
print(f"Shape: {df_occupation_clean.shape}")
print(f"Missing values: {df_occupation_clean.isnull().sum().sum()}")

df_occupation_clean.to_csv(f"{OUT_PATH}/occupation_master.csv", index=False)
print(f"Saved to: {OUT_PATH}/occupation_master.csv")

=== BEFORE CLEANING ===
Shape: (1016, 3)
Missing values: 0
Duplicates: 0

=== AFTER CLEANING ===
Shape: (1016, 3)
Missing values: 0


Saved to: ../data/processed/occupation_master.csv


---
## 4. Cleaning essential_skills.csv
---

In [11]:
print("=== BEFORE CLEANING ===")
print(f"Shape: {df_essential.shape}")
print(f"Missing values: {df_essential.isnull().sum().sum()}")
print(f"Duplicates: {df_essential.duplicated().sum()}")

df_essential_clean = df_essential.copy()

# Check missing values
missing = df_essential_clean.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Missing values:")
    print(missing)
else:
    print("No missing values.")

=== BEFORE CLEANING ===
Shape: (18200, 15)
Missing values: 9100
Duplicates: 0
Missing values:
Not Relevant    9100
dtype: int64


In [12]:
# Remove duplicates
dup_count = df_essential_clean.duplicated().sum()
if dup_count > 0:
    df_essential_clean = df_essential_clean.drop_duplicates()
    print(f"Removed {dup_count} duplicates. Shape: {df_essential_clean.shape}")

# Normalize skill/element names
# Strip whitespace from key columns
for col in ["O*NET-SOC Code", "Title", "Element Name"]:
    if col in df_essential_clean.columns:
        df_essential_clean[col] = df_essential_clean[col].str.strip()

# Normalize skill names - remove leading/trailing whitespace, standardize spacing
df_essential_clean["Element Name"] = df_essential_clean["Element Name"].str.replace("\\s+", " ", regex=True).str.strip()

# Drop rows where critical columns are missing
critical_cols = ["O*NET-SOC Code", "Element Name"]
before_count = len(df_essential_clean)
df_essential_clean = df_essential_clean.dropna(subset=[c for c in critical_cols if c in df_essential_clean.columns])
after_count = len(df_essential_clean)
print(f"Removed {before_count - after_count} rows with missing critical columns.")

print()
print("=== AFTER CLEANING ===")
print(f"Shape: {df_essential_clean.shape}")

df_essential_clean.to_csv(f"{OUT_PATH}/essential_skills_processed.csv", index=False)
print(f"Saved to: {OUT_PATH}/essential_skills_processed.csv")

Removed 0 rows with missing critical columns.

=== AFTER CLEANING ===
Shape: (18200, 15)


Saved to: ../data/processed/essential_skills_processed.csv


---
## 5. Cleaning software_skills.csv
---

In [13]:
print("=== BEFORE CLEANING ===")
print(f"Shape: {df_software.shape}")
print(f"Missing values: {df_software.isnull().sum().sum()}")
print(f"Duplicates: {df_software.duplicated().sum()}")

df_software_clean = df_software.copy()

# Check missing values
missing = df_software_clean.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Missing values:")
    print(missing)
else:
    print("No missing values.")

=== BEFORE CLEANING ===
Shape: (31821, 7)
Missing values: 0


Duplicates: 0
No missing values.


In [14]:
# Remove duplicates
dup_count = df_software_clean.duplicated().sum()
if dup_count > 0:
    df_software_clean = df_software_clean.drop_duplicates()
    print(f"Removed {dup_count} duplicates. Shape: {df_software_clean.shape}")

# Normalize string columns
for col in df_software_clean.select_dtypes(include=["object", "str"]).columns:
    df_software_clean[col] = df_software_clean[col].str.strip()

# Normalize software/skill names
if "Workplace Example" in df_software_clean.columns:
    df_software_clean["Workplace Example"] = (
        df_software_clean["Workplace Example"].str.replace("\\s+", " ", regex=True).str.strip()
    )
if "Element Name" in df_software_clean.columns:
    df_software_clean["Element Name"] = (
        df_software_clean["Element Name"].str.replace("\\s+", " ", regex=True).str.strip()
    )

# Drop rows where critical columns are missing
critical_cols = ["O*NET-SOC Code", "Element Name"]
before_count = len(df_software_clean)
df_software_clean = df_software_clean.dropna(subset=[c for c in critical_cols if c in df_software_clean.columns])
after_count = len(df_software_clean)
print(f"Removed {before_count - after_count} rows with missing critical columns.")

print()
print("=== AFTER CLEANING ===")
print(f"Shape: {df_software_clean.shape}")

df_software_clean.to_csv(f"{OUT_PATH}/software_skills_processed.csv", index=False)
print(f"Saved to: {OUT_PATH}/software_skills_processed.csv")

Removed 0 rows with missing critical columns.

=== AFTER CLEANING ===
Shape: (31821, 7)
Saved to: ../data/processed/software_skills_processed.csv


---
## Cleaning Summary

| Dataset | Original Shape | Cleaned Shape | Actions |
| --- | --- | --- | --- |
| employee_attrition | 1470 x 35 | ~1470 x 32 | Dropped 3 constant columns, normalized categories |
| hr_performance_engagement | varies | varies | Handled missing values, removed duplicates |
| occupation_data | varies | varies | Removed duplicates, normalized strings |
| essential_skills | varies | varies | Removed duplicates, normalized skill names |
| software_skills | varies | varies | Removed duplicates, normalized software names |

All cleaned datasets saved to `data/processed/`.

---
**Next step:** Data Relationships (04_data_relationships.ipynb)